# Project 1: Create a simple Podacast

Authors: Akansha Verma & Louise Plessis

In [22]:
import os
import re
import io
from pathlib import Path
from dataclasses import dataclass

import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

DATA_DIR = Path("input_data")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL = "gpt-4o-mini"

Key loaded: True


## Scrapper

In [23]:
# Data structure & constants (run once)
@dataclass
class SourceDocument:
    source_type: str
    origin: str
    raw_text: str


HEADERS = {
    # Some sites block requests with no User-Agent; a normal browser UA avoids that.
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

# Tags that are almost never part of the actual article content.
NOISE_TAGS = ["script", "style", "nav", "header", "footer", "aside", "form", "noscript"]

# Functions (run once)
def _slugify(url: str) -> str:
    """Turns a URL into a safe filename, e.g. mindfulambition-net-beginners-mind.txt"""
    slug = re.sub(r"^https?://", "", url)
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", slug).strip("-").lower()
    return slug[:100]


def save_to_file(doc: SourceDocument, folder: Path = DATA_DIR) -> Path:
    """Saves a SourceDocument's raw_text to 'input_data/<slug>.txt' and returns the path."""
    filename = f"{_slugify(doc.origin)}.txt"
    out_path = folder / filename
    out_path.write_text(doc.raw_text, encoding="utf-8")
    return out_path


def load_from_url(url: str, timeout: int = 15) -> SourceDocument:
    """Downloads the page at `url` and extracts readable article text."""
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(NOISE_TAGS):
        tag.decompose()

    container = (
        soup.find("article")
        or soup.find("div", class_=lambda c: c and "entry-content" in c)
        or soup.find("div", class_=lambda c: c and "post-content" in c)
        or soup.find("main")
    )
    paragraphs = container.find_all("p") if container else soup.find_all("p")

    text = "\n\n".join(
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    )

    if not text:
        raise ValueError(f"No article text found at {url} — page structure may need a custom selector.")

    return SourceDocument(source_type="url", origin=url, raw_text=text)

# Run the scrapper per URLs
doc = load_from_url("https://mindfulambition.net/beginners-mind/")
saved_path = save_to_file(doc)

print(f"Source: {doc.origin}")
print(f"Extracted {len(doc.raw_text)} characters, {len(doc.raw_text.split())} words")
print(f"Saved to: {saved_path}")
print("---")
print(doc.raw_text[:300], "...")

urls = [
    "https://mindfulambition.net/beginners-mind/",
    "https://www.infoprolearning.com/blog/understanding-the-learning-curve-why-its-important-in-employee-training-and-development/",
    "https://iopn.library.illinois.edu/pressbooks/instructioninlibraries/chapter/learning-theories-understanding-how-people-learn/",
    "https://www.cmu.edu/teaching/designteach/design/instructionalstrategies/groupprojects/benefits.html",
]

for url in urls:
    doc = load_from_url(url)
    path = save_to_file(doc)
    print(f"✓ {url} -> {path} ({len(doc.raw_text.split())} words)")

def load_from_text_file(path: str) -> SourceDocument:
    text = Path(path).read_text(encoding="utf-8")
    return SourceDocument(source_type="text", origin=str(path), raw_text=text)

def load_all_from_folder(folder: str = "input_data", pattern: str = "*.txt") -> list:
    """Reads every .txt file in `folder` and returns one SourceDocument per file."""
    folder_path = Path(folder)
    files = sorted(folder_path.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No .txt files found in {folder_path.resolve()}")
    return [load_from_text_file(str(f)) for f in files]

Source: https://mindfulambition.net/beginners-mind/
Extracted 4251 characters, 715 words
Saved to: input_data/mindfulambition-net-beginners-mind.txt
---
I used to commute to downtown Chicago by train every day.

At first, I enjoyed this new facet of life. Thenoveltyof the experience made it fun and exciting.

But that fresh feeling quickly washed away, turning my commute into a repetitive inconvenience.All this extra time spent getting from one plac ...
✓ https://mindfulambition.net/beginners-mind/ -> input_data/mindfulambition-net-beginners-mind.txt (715 words)
✓ https://www.infoprolearning.com/blog/understanding-the-learning-curve-why-its-important-in-employee-training-and-development/ -> input_data/www-infoprolearning-com-blog-understanding-the-learning-curve-why-its-important-in-employee-training.txt (845 words)
✓ https://iopn.library.illinois.edu/pressbooks/instructioninlibraries/chapter/learning-theories-understanding-how-people-learn/ -> input_data/iopn-library-illinois-edu-pres

## Process sources

In [24]:
CONDENSE_SYSTEM_PROMPT="You are the witty, warm solo host of a podcast called 'The Joy of Learning.' Your job is to turn source material into an engaging spoken-word script — not a summary, a script meant to be read aloud by a text-to-speech voice.\n\nRules:\n1. Open with exactly this greeting style: 'Welcome to The Joy of Learning, the podcast where we explore [general subject in your own words].' Then transition naturally into the episode.\n2. Explain the core concept in your own words — do not summarize the source paragraph by paragraph, and do not reference the article, its author, or where the content came from. Write as if this is your own idea you're excited to share.\n3. Be genuinely funny: use a playful analogy, a light joke, or a self-aware aside at least once. Think 'smart friend explaining something cool over coffee,' not 'lecture.'\n4. Write only what should be spoken aloud — no headers, no stage directions, no markdown, no sound-effect cues.\n5. Keep it under 300 words so the TTS output stays a reasonable length.\n6. End with a short, warm sign-off that invites the listener to try adopting the mindset themselves."
def condense_source(doc: SourceDocument, max_chars: int = 12000) -> str:
    text = doc.raw_text[:max_chars]
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": CONDENSE_SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        temperature=0.8,
    )
    return response.choices[0].message.content

SCRIPT_SYSTEM_PROMPT = """You are the witty, warm solo host of a podcast called
"The Joy of Learning." Your job is to weave several research briefs into ONE
continuous, engaging spoken-word episode script — not a summary, a script meant
to be read aloud by a text-to-speech voice.

Rules:
1. Open with exactly this greeting style: "Welcome to The Joy of Learning, the
   podcast where we explore [general subject in your own words]." Then
   transition naturally into the episode.
2. Weave the ideas from ALL the briefs into a single narrative arc with smooth
   transitions between them — do not treat them as separate segments.
3. Explain concepts in your own words. Never reference "the article," "the
   source," or where the content came from.
4. Be genuinely funny at least once every couple of minutes of runtime.
5. Write only what should be spoken aloud — no headers, no markdown, no
   stage directions.
6. End with a short, warm sign-off.
7. Target roughly {target_words} words total for {num_sources} ideas."""

def write_episode_script(condensed_sources: dict, episode_title, topic, audience, target_words=1300) -> str:
    system_prompt = SCRIPT_SYSTEM_PROMPT.format(target_words=target_words, num_sources=len(condensed_sources))
    briefs_block = "\n\n".join(
        f"--- Brief {i+1} (from {origin}) ---\n{notes}"
        for i, (origin, notes) in enumerate(condensed_sources.items())
    )
    user_prompt = (
        f"Episode title: {episode_title}\n"
        f"Episode topic: {topic}\n"
        f"Target audience: {audience}\n\n"
        f"Here are the research briefs to weave together:\n\n{briefs_block}"
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.8,
    )
    return response.choices[0].message.content


def generate_episode_script(docs, episode_title, topic, audience, target_words=1300):
    """Full transformation step: condense every doc, then write the unified script."""
    condensed = {doc.origin: condense_source(doc) for doc in docs}
    return write_episode_script(condensed, episode_title, topic, audience, target_words)


In [25]:
MAX_CHARS = 3800  # safety margin under the ~4096-char TTS input limit

def chunk_text(text: str, max_chars: int = MAX_CHARS) -> list:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], ""
    for sentence in sentences:
        candidate = f"{current} {sentence}".strip()
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


def synthesize_chunk(text: str, voice: str = "alloy", model: str = "gpt-4o-mini-tts") -> bytes:
    response = client.audio.speech.create(model=model, voice=voice, input=text)
    return response.content


def _concatenate_mp3(audio_chunks: list) -> bytes:
    """Uses pydub+ffmpeg for a clean join if available, else raw byte concat."""
    try:
        from pydub import AudioSegment
        combined = AudioSegment.empty()
        for chunk_bytes in audio_chunks:
            combined += AudioSegment.from_file(io.BytesIO(chunk_bytes), format="mp3")
        buffer = io.BytesIO()
        combined.export(buffer, format="mp3")
        return buffer.getvalue()
    except Exception as e:
        print(f"pydub/ffmpeg unavailable ({e}); falling back to raw byte concatenation.")
        return b"".join(audio_chunks)


def text_to_speech(text: str, output_file: str = "output/episode.mp3", voice: str = "alloy") -> str:
    chunks = chunk_text(text)
    print(f"Synthesizing {len(chunks)} chunk(s)...")
    audio_bytes_list = [synthesize_chunk(c, voice=voice) for c in chunks]
    combined_audio = _concatenate_mp3(audio_bytes_list)

    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(combined_audio)
    print(f"Audio saved to {out_path}")
    return str(out_path) 

In [27]:
docs = load_all_from_folder("input_data")
print(f"Loaded {len(docs)} source(s):", [d.origin for d in docs])

script = generate_episode_script(
    docs,
    episode_title="The Joy of Learning: Embracing the Curve",
    topic="the learning curve, cognitive science, beginner's mind, and learning in groups",
    audience="adult learners currently going through a steep learning curve",
    target_words=1300,
)
print(script)

Loaded 5 source(s): ['input_data/iopn-library-illinois-edu-pressbooks-instructioninlibraries-chapter-learning-theories-understanding-.txt', 'input_data/mindfulambition-net-beginners-mind.txt', 'input_data/www-cmu-edu-teaching-designteach-design-instructionalstrategies-groupprojects-benefits-html.txt', 'input_data/www-infoprolearning-com-blog-understanding-the-learning-curve-why-its-important-in-employee-training.txt', 'input_data/www-youtube-com-watch-v-awukng1upes.txt']
Welcome to The Joy of Learning, the podcast where we explore the fascinating world of how we acquire knowledge! Today, let’s take a delightful dive into the concept of the learning curve, embracing the mindset of a beginner, and the joys of collaboration. So grab your imaginary helmets and knee pads because we’re about to embark on a steep but thrilling ride!

Let’s start with this little gem called the learning curve. Picture it: you’re trying to master a new skill, maybe juggling, or perhaps something more practical 

## Create Audio

In [28]:
audio_path = text_to_speech(script, output_file="output/episode.mp3", voice="alloy")
audio_path

Synthesizing 2 chunk(s)...
Audio saved to output/episode.mp3


'output/episode.mp3'

In [ ]:
import gradio as gr
import traceback

VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]


def generate_podcast(folder_path, episode_title, topic, audience, voice, target_words, progress=gr.Progress()):
    log_lines = []

    progress(0.0, desc="Loading sources...")
    try:
        docs = load_all_from_folder(folder_path or "input_data")
    except FileNotFoundError as e:
        raise gr.Error(str(e))

    for doc in docs:
        log_lines.append(f"Loaded: {doc.origin} [{len(doc.raw_text.split())} words]")
    progress(0.3, desc=f"Loaded {len(docs)} file(s)")

    try:
        script = generate_episode_script(
            docs, episode_title or "Untitled Episode", topic or "the episode's subject",
            audience or "general listeners", int(target_words),
        )
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Script generation failed: {e}")

    progress(0.75, desc="Generating audio...")
    safe_title = "".join(c if c.isalnum() or c in " -_" else "" for c in (episode_title or "episode"))
    try:
        audio_path = text_to_speech(script, output_file=f"output/{safe_title or 'episode'}.mp3", voice=voice)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Audio generation failed: {e}")

    progress(1.0, desc="Done!")
    return script, audio_path, "\n".join(log_lines)


with gr.Blocks(title="The Joy of Learning — Podcast Studio", theme=theme, css=custom_css) as demo:
    gr.HTML(
        """
        <div id="header-banner">
            <h1>The Joy of Learning</h1>
            <p>Turn articles & videos into a warm, witty podcast episode — automatically.</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            sources_input = gr.Textbox(
                label="📥 Sources (one per line — article or YouTube URLs)",
                lines=8,
                placeholder="https://example.com/article\nhttps://youtube.com/watch?v=...",
            )
            voice_input = gr.Dropdown(VOICES, value="alloy", label="🎤 Voice")
            generate_btn = gr.Button("✨ Generate Podcast", variant="primary", elem_classes="generate-btn")

        with gr.Column(scale=1):
            title_output = gr.Textbox(label="📝 Generated Episode Title", interactive=False)
            audio_output = gr.Audio(label="🔊 Episode Audio", type="filepath")
            script_output = gr.Textbox(label="Episode Script", lines=12)
            log_output = gr.Textbox(label="Pipeline Log", lines=6)

    generate_btn.click(
        fn=generate_podcast,
        inputs=[sources_input],
        outputs=[title_output, script_output, audio_output, log_output],
    )
if __name__ == "__main__":
    Path("output").mkdir(exist_ok=True)
    demo.launch()



/var/folders/7d/mw8sk41d1rb_hx7v63zn3v180000gn/T/ipykernel_37203/4064652373.py:41: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="The Joy of Learning — Podcast Studio", theme=theme, css=custom_css) as demo:
/opt/miniconda3/envs/bootcamp-env/lib/python3.13/site-packages/gradio/utils.py:1220: UserWarning: Expected at least 6 arguments for function <function generate_podcast at 0x1627e2660>, received 1.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


/opt/miniconda3/envs/bootcamp-env/lib/python3.13/site-packages/gradio/helpers.py:1081: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
Traceback (most recent call last):
  File "/var/folders/7d/mw8sk41d1rb_hx7v63zn3v180000gn/T/ipykernel_37203/4064652373.py", line 23, in generate_podcast
    audience or "general listeners", int(target_words),
                                     ~~~^^^^^^^^^^^^^^
TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
Traceback (most recent call last):
  File "/var/folders/7d/mw8sk41d1rb_hx7v63zn3v180000gn/T/ipykernel_37203/4064652373.py", line 23, in generate_podcast
    audience or "general listeners", int(target_words),
                                     ~~~^^^^^^^^^^^^^^
TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

During handling of the above exception, another exception occurred:

Traceback